In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import zipfile
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import numpy as np
import time
import copy
from tqdm import tqdm

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# Create a directory for the dataset
os.makedirs('/content/catdog', exist_ok=True)

# Paths to your zip files in Drive (adjust if you placed them elsewhere)
train_zip = '/content/drive/MyDrive/cat-dog-dataset/training_set.zip'
test_zip = '/content/drive/MyDrive/cat-dog-dataset/test_set.zip'

# Unzip training set
with zipfile.ZipFile(train_zip, 'r') as zip_ref:
    zip_ref.extractall('/content/catdog/training')

# Unzip test set
with zipfile.ZipFile(test_zip, 'r') as zip_ref:
    zip_ref.extractall('/content/catdog/test')

print("Unzipped!")

Unzipped!


In [ ]:
# Transform for pretrained models (normalize with ImageNet stats)
pretrained_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Transform for simple CNN (no normalization, just resize & tensor)
simple_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Load dataset (assuming folder structure: train/cats, train/dogs)
full_dataset = datasets.ImageFolder(root='/content/catdog/training/training_set', transform=pretrained_transform)  # we'll change transform per model

# Split into train/validation (80/20)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# For the simple CNN we need to override the transform
train_dataset.dataset.transform = simple_transform
val_dataset.dataset.transform = simple_transform

# Create data loaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

dataloaders = {'train': train_loader, 'val': val_loader}
dataset_sizes = {'train': len(train_dataset), 'val': len(val_dataset)}

# Simple CNN

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=2):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 256),  # after 3 pools: 224/2/2/2 = 28
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Pretrained ResNet18

In [ ]:
def get_resnet18(num_classes=2):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    # Freeze all layers (optional – we can also fine-tune)
    for param in model.parameters():
        param.requires_grad = False
    # Replace the classifier
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    return model

# Pretrained VGG16

In [ ]:
def get_vgg16(num_classes=2):
    model = models.vgg16_bn(weights=models.VGG16_BN_Weights.IMAGENET1K_V1)
    for param in model.parameters():
        param.requires_grad = False
    # Replace classifier
    num_ftrs = model.classifier[6].in_features
    model.classifier[6] = nn.Linear(num_ftrs, num_classes)
    return model

# Training Function

In [ ]:
def train_model(model, criterion, optimizer, scheduler=None, num_epochs=10):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train' and scheduler:
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed//60:.0f}m {time_elapsed%60:.0f}s')
    print(f'Best val Acc: {best_acc:.4f}')

    model.load_state_dict(best_model_wts)
    return model, history

# Train Each Model

In [ ]:
models_to_train = {
    'SimpleCNN': SimpleCNN(num_classes=2),
    'ResNet18 (pretrained)': get_resnet18(),
    'VGG16 (pretrained)': get_vgg16()
}

results = {}

for name, model in models_to_train.items():
    print(f"\n===== Training {name} =====")
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)
    # For pretrained models we only train the new head, so lr can be higher
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

    trained_model, history = train_model(model, criterion, optimizer, scheduler, num_epochs=10)
    results[name] = history

# Comparison and Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

metrics = ['loss', 'acc']
phases = ['train', 'val']
colors = {'SimpleCNN': 'blue', 'ResNet18 (pretrained)': 'green', 'VGG16 (pretrained)': 'red'}

for i, metric in enumerate(metrics):
    for phase in phases:
        ax = axes[i*2 + (0 if phase=='train' else 1)]
        ax.set_title(f'{phase} {metric}')
        ax.set_xlabel('epoch')
        ax.set_ylabel(metric)
        for name, hist in results.items():
            ax.plot(hist[f'{phase}_{metric}'], label=name, color=colors[name])
        ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
print("Model Comparison:")
for name, hist in results.items():
    final_val_acc = hist['val_acc'][-1]
    print(f"{name}: final val acc = {final_val_acc:.4f}")

In [ ]:
from tqdm import tqdm  # already imported

def train_model(model, criterion, optimizer, scheduler=None, num_epochs=10):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch+1}/{num_epochs}')
        print('-' * 60)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                # Create tqdm progress bar for training data
                data_loader = tqdm(dataloaders[phase],
                                   desc=f'Training',
                                   leave=False)  # leave=False clears bar after epoch
            else:
                model.eval()
                data_loader = tqdm(dataloaders[phase],
                                   desc=f'Validation',
                                   leave=False)

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in data_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # Update statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

                # Update tqdm postfix with current loss and accuracy for this batch
                data_loader.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'acc': f'{torch.sum(preds == labels.data).item()/inputs.size(0):.4f}'
                })

            if phase == 'train' and scheduler:
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())

            # Print epoch summary (tqdm automatically handles new line)
            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

    time_elapsed = time.time() - since
    print(f'\nTraining complete in {time_elapsed//60:.0f}m {time_elapsed%60:.0f}s')
    print(f'Best val Acc: {best_acc:.4f}')

    model.load_state_dict(best_model_wts)
    return model, history